# Import

In [60]:
import math
import torch
import torch.nn as nn

# The Problem in previous models version

- I love AI

- AI Love I

# both same so to solve that we use positional Encoding

- Simple every positions gets unique Vectors


________________________________________________________

# This one Transformer Layer

Input

|

MultiheadAttention

|

Add(Residual) (just adding x + attention_output) 

|

LayerNorm (just a normalization layer)

|

Feed Forward Network

|

Add (Residual)

|

LayerNorm
___________________________________________________________________________

# But Real Transformer 

Layer 1 (Encoder)

 ↓

Layer 2 (Encoder)

 ↓

Layer 3 (Encoder)

 ↓

Layer 4 (Encoder)

 ...

### They stay encoder Blocks

In [61]:
# but why

# Layer 1 (Learns Basic Relationships)
#  ↓
# Layer 2 (More Semantic Relationships)
#  ↓
# Layer 3 (Sentence level-Understanding)
#  ↓
# Layer 4 (High-level Understanding)
#  ...

# V8 Positional Encoding

In [62]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len = 5000):

        super().__init__()

        # Empty matrix
        pe = torch.zeros(max_len, d_model)

        # create positions
        positions = torch.arange(0, max_len).unsqueeze(-1)
        #         same sa np.arange           convert row to column

        # create denominator 
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        #      0 to n by 2 step(0, 2, 4, 6)            log 10000 -> 9.210  / 8 -> 1.151 -> -1.151
        )

        # Fill even dimensions
        pe[:, 0::2] = torch.sin(positions * div_term)

        # Fill odd dimensions
        pe[:, 1::2] = torch.cos(positions * div_term)

        # Add batch dimension
        pe = pe.unsqueeze(0) # add extra bracket (1, 5000, 8)

        # Register buffer
        self.register_buffer("pe", pe) #why? Not trainable, But moves with model

    def forward(self, x):
        seq_len = x.size(1)

        return (x + self.pe[:, :seq_len])



In [63]:
positions = torch.arange(0, 10).unsqueeze(-1)
positions

tensor([[0],
        [1],
        [2],
        [3],
        [4],
        [5],
        [6],
        [7],
        [8],
        [9]])

In [64]:
class MultiHeadAttention(nn.Module):
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        
        # assert is keyword
        # its a debugging tools used to test if a specific condition in our code evalutes to true
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads


        self.Wq = nn.Linear(in_features = d_model, out_features = d_model)
        self.Wk = nn.Linear(in_features = d_model, out_features = d_model)
        self.Wv = nn.Linear(in_features = d_model, out_features = d_model)

        self.out_proj = nn.Linear(in_features = d_model, out_features = d_model)

    def forward(self, x):

        batch_size, seq_len, d_model = x.shape

        print('=' * 10)
        print("x shape ", x.shape)

        #send x in linear layer

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        print("\n After Linear Layers")

        print("Q : ", Q.shape)
        print("K : ", K.shape)
        print("V : ", V.shape)

        # split into heads using view
        # view is like reshaping but without copy 

        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )
        

        print("\n After reshaping")
        print(Q.shape)
        print(K.shape)
        print(V.shape)

        # Move heads forward
        
        # change shape 
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        print("\n After Transpose")
        print("Q : ", Q.shape)
        print("K : ", K.shape)
        print("V : ", V.shape)

        # Attention score

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)

        print("\nscore")
        print(scores.shape)

        # softmax

        weights = torch.softmax(scores, dim = -1)

        print("\nAfter softmax Weight Shape")
        print(weights.shape)

        print("\nHead 1 Attentation matrix Weights")
        print(weights[0, 0])
        print("\nHead 2 Attentation matrix Weights")
        print(weights[0, 1])
        
        # Apply Attention
        output = weights @ V

        print("\nAfter Weights @ V")
        print(output.shape)

        # Combine heads
        output = output.transpose(1, 2)

        print("\nAfter Transpose Back")
        print(output.shape)

        # After Transpose we can't use view to reshape it 
        # so contiguous allocates a new block of memory to copy and rearrange a tensor's data into a sequential, unbroken memory layout.
        # then apply view()

        # print(output.contiguous())

        output = output.contiguous().view(
            batch_size,
            seq_len,
            d_model
        )

        # After Flatten
        print("\nAfter Flatten heads (2, 4 to 8)")
        print(output.shape)

        # Final projection

        output = self.out_proj(output)

        print("\nFinal output")
        print(output.shape)

        return output 

In [65]:
class FeedForwardNetwork(nn.Module):

    def __init__(self, d_model):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(in_features = d_model, out_features = d_model * 4),

            nn.GELU(),

            nn.Linear(in_features = d_model * 4, out_features = d_model)
        )

    def forward(self, x):
        return self.net(x)


In [66]:
class TransformerEncoderBlock(nn.Module):
    
    def __init__(self, d_model, num_heads):

        super().__init__()

        # created a obj for MHA
        self.mha = MultiHeadAttention(d_model = d_model, num_heads = num_heads)

        # LayerNorm from nn (new thing in V5)
        self.norm1 = nn.LayerNorm(d_model)

        # V6

        # created a obj for FFN
        self.ffn = FeedForwardNetwork(d_model)

        # LayerNorm
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        print("\n Input x")
        print(x)

        # giving value for the obj
        attention_output = self.mha(x)

        print("\nAttention output")
        print(attention_output.shape)

        # residual
        x += attention_output # Residual (new thing in V5)

        print("\nresidual output")
        print(x.shape)

        # normalization
        x = self.norm1(x)

        print("\nNormalization (LayerNorm)")
        print(x.shape)

        # FFN Block

        ffn_output = self.ffn(x)

        print("\n After FFN")
        print(ffn_output.shape)

        # Residual
        x += ffn_output

        print("\nAfter Add Residual")
        print(x.shape)

        # Normalization

        x = self.norm2(x)
        
        print("\nafter LayerNorm")
        print(x.shape)


        return x



# V7 Multiple TransformerEncoderBlocks in TransformerEncoder

In [67]:
class TransformerEncoder(nn.Module):

    def __init__(self, d_model, num_heads, num_layers):

        super().__init__()
        # Why ModuleList()
        # Suppose num_layers = 4
        # block-1, block-2, block-3 and block-4
        # stored inside -> self.layers
        # Pytorch can then track all parameters

        self.layers = nn.ModuleList([
            TransformerEncoderBlock(
                d_model = d_model,
                num_heads = num_heads
            )

            for _ in range(num_layers)
        ])

    def forward(self, x):
        
        for i, layer in enumerate(self.layers):
            
            print(f"\n ====== Encoder Layer {i+1} ======")

            x = layer(x)

        return x


In [68]:
# Example input

batch_size = 1
seq_len = 3
d_model = 8
num_heads = 2

vocab = {
    "I" : 0,
    "Love" : 1,
    "AI" : 2
}   

tokens = torch.tensor([0, 1, 2]) # or torch.tensor(vocab.values())

embedding = nn.Embedding(
    num_embeddings = len(vocab),
    embedding_dim = d_model
)

x  = embedding(tokens)

x = x.view(1, 3, 8)


pos_encoder = PositionalEncoding(d_model = d_model)

x_positional = pos_encoder(x)

print("\npositional encoded x")
print(x_positional.shape)

tfe = TransformerEncoder(d_model = d_model, num_heads = num_heads, num_layers = 4)

output = tfe(x_positional)

print("\noutput")
print(output.shape)


positional encoded x
torch.Size([1, 3, 8])

 ====== Encoder Layer 1 ======

 Input x
tensor([[[-0.4449,  2.1331, -0.1850,  2.7518,  0.7807,  1.0631, -0.7383,
          -0.3275],
         [ 1.7565,  0.8787, -0.7774,  0.5550,  1.1297,  1.7660,  0.0089,
           2.1096],
         [ 1.4641, -1.5808, -1.4512,  1.9381,  0.1313,  1.8138, -1.2487,
          -0.2749]]], grad_fn=<AddBackward0>)
x shape  torch.Size([1, 3, 8])

 After Linear Layers
Q :  torch.Size([1, 3, 8])
K :  torch.Size([1, 3, 8])
V :  torch.Size([1, 3, 8])

 After reshaping
torch.Size([1, 3, 2, 4])
torch.Size([1, 3, 2, 4])
torch.Size([1, 3, 2, 4])

 After Transpose
Q :  torch.Size([1, 2, 3, 4])
K :  torch.Size([1, 2, 3, 4])
V :  torch.Size([1, 2, 3, 4])

score
torch.Size([1, 2, 3, 3])

After softmax Weight Shape
torch.Size([1, 2, 3, 3])

Head 1 Attentation matrix Weights
tensor([[0.4183, 0.2413, 0.3404],
        [0.3556, 0.2423, 0.4021],
        [0.4045, 0.1363, 0.4592]], grad_fn=<SelectBackward0>)

Head 2 Attentation matr

In [69]:
print(pos_encoder.pe[0, :3])

tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00]])


In [70]:
print(pos_encoder.pe[0, :5])

tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
          9.9955e-01,  3.0000e-03,  1.0000e+00],
        [-7.5680e-01, -6.5364e-01,  3.8942e-01,  9.2106e-01,  3.9989e-02,
          9.9920e-01,  4.0000e-03,  9.9999e-01]])
